# 46 Bank Reconciliation Agent

**Pattern**: Deterministic pre-matching + LLM exception classification  
**Framework**: Raw `openai` SDK (no LangGraph)  
**Key idea**: LLM call volume scales with the number of exceptions, not total transaction volume.

In [ ]:
%pip install -q openai pydantic python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Part 1 — Business Problem

Month-end bank reconciliation compares every line of the **bank statement** against
the **General Ledger (GL) cash account**. The accountant must explain every
difference and certify that the two balances agree after adjustments.

Most differences are routine:
- **Timing differences** — a cheque recorded in the GL has not yet cleared the bank.
- **Bank charges** — a service fee applied by the bank has no automatic GL entry.

The hard items are genuine exceptions: unusual wires, possible duplicates, or fraud
indicators. These are few in number but high in risk.

**Agent design goal**: resolve the routine items deterministically (no LLM cost),
then send only the genuine exceptions to the LLM for classification.

## Part 2 — Schema

The reconciliation uses five Pydantic models:

| Model | Purpose |
|-------|---------|
| `BankTransaction` | One line from the bank statement |
| `GLEntry` | One line from the GL cash account |
| `MatchedPair` | A confirmed bank ↔ GL match with confidence level |
| `UnmatchedItem` | An item needing human review, classified by the LLM |
| `BankReconciliationSummary` | Full output: matches, exceptions, status, note |

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field


class BankTransaction(BaseModel):
    txn_id: str = Field(description="Unique bank transaction ID")
    date: str = Field(description="Date in YYYY-MM-DD format")
    description: str = Field(description="Narrative from the bank statement")
    debit: float = Field(default=0.0, description="Outflow amount")
    credit: float = Field(default=0.0, description="Inflow amount")


class GLEntry(BaseModel):
    entry_id: str = Field(description="Unique GL entry ID")
    date: str = Field(description="Posting date in YYYY-MM-DD format")
    reference: str = Field(description="Journal reference")
    amount: float = Field(description="Signed amount: positive=debit, negative=credit")
    description: str = Field(description="GL narrative")


class MatchedPair(BaseModel):
    bank_txn_id: str
    gl_entry_id: str
    match_confidence: Literal["exact", "probable", "fuzzy"]


class UnmatchedItem(BaseModel):
    item_id: str
    source: Literal["bank", "gl"]
    amount: float
    description: str
    exception_type: Literal[
        "timing_difference", "bank_charge", "duplicate",
        "missing_booking", "fraud_indicator"
    ]
    recommended_action: str


class BankReconciliationSummary(BaseModel):
    period: str
    bank_closing_balance: float
    gl_cash_balance: float
    matched_pairs: List[MatchedPair]
    unmatched_items: List[UnmatchedItem]
    reconciling_items_value: float
    is_reconciled: bool
    reconciliation_note: str


print("Schema defined — 5 models ready.")

## Part 3 — Deterministic Pre-Matcher

### Why deterministic pre-matching matters

A naive approach would send every bank transaction to the LLM and ask it to
find the matching GL entry. For a company with 500 monthly transactions,
that means 500 LLM calls every month — expensive, slow, and unnecessary.

The pre-matcher resolves the easy cases with simple arithmetic:

| Pass | Rule | Confidence |
|------|------|------------|
| 1 | Same absolute amount AND same date | `exact` |
| 2 | Same absolute amount AND date within 2 days | `probable` |

In practice, passes 1 and 2 resolve ~90–95 % of items. The LLM only sees
the residual exceptions — typically fewer than 10 per month.

This is the **bounded matching loop** pattern: LLM cost is proportional to
the problem, not the data size.

In [ ]:
from datetime import datetime
from typing import Dict, Tuple


def date_diff_days(d1: str, d2: str) -> int:
    fmt = "%Y-%m-%d"
    return abs((datetime.strptime(d1, fmt) - datetime.strptime(d2, fmt)).days)


def amounts_match(a: float, b: float, tol: float = 0.01) -> bool:
    return abs(abs(a) - abs(b)) <= tol


def find_matches(
    bank_txns: List[Dict], gl_entries: List[Dict]
) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    matched_raw: List[Dict] = []
    used_bank: set = set()
    used_gl: set = set()

    def _bank_amount(txn: Dict) -> float:
        return txn.get("credit", 0.0) - txn.get("debit", 0.0)

    # Pass 1 — exact
    for bank in bank_txns:
        for gl in gl_entries:
            if bank["txn_id"] in used_bank or gl["entry_id"] in used_gl:
                continue
            if amounts_match(_bank_amount(bank), gl["amount"]) and bank["date"] == gl["date"]:
                matched_raw.append({"bank_txn_id": bank["txn_id"], "gl_entry_id": gl["entry_id"], "match_confidence": "exact"})
                used_bank.add(bank["txn_id"])
                used_gl.add(gl["entry_id"])
                break

    # Pass 2 — probable (within 2 days)
    for bank in bank_txns:
        if bank["txn_id"] in used_bank:
            continue
        for gl in gl_entries:
            if gl["entry_id"] in used_gl:
                continue
            if amounts_match(_bank_amount(bank), gl["amount"]) and date_diff_days(bank["date"], gl["date"]) <= 2:
                matched_raw.append({"bank_txn_id": bank["txn_id"], "gl_entry_id": gl["entry_id"], "match_confidence": "probable"})
                used_bank.add(bank["txn_id"])
                used_gl.add(gl["entry_id"])
                break

    unmatched_bank = [b for b in bank_txns if b["txn_id"] not in used_bank]
    unmatched_gl = [g for g in gl_entries if g["entry_id"] not in used_gl]
    return matched_raw, unmatched_bank, unmatched_gl


# Quick smoke test
test_bank = [
    {"txn_id": "T1", "date": "2025-06-01", "description": "Payment", "debit": 0.0, "credit": 500.0},
    {"txn_id": "T2", "date": "2025-06-03", "description": "Fee", "debit": 12.0, "credit": 0.0},
]
test_gl = [
    {"entry_id": "G1", "date": "2025-06-01", "reference": "JE-1", "amount": 500.0, "description": "Cash receipt"},
]
m, ub, ug = find_matches(test_bank, test_gl)
print(f"Matched: {len(m)}, Unmatched bank: {len(ub)}, Unmatched GL: {len(ug)}")
assert m[0]["match_confidence"] == "exact"
assert ub[0]["txn_id"] == "T2"
print("Pre-matcher smoke test passed.")

## Part 4 — Workflow

The `run()` function wires together the pre-matcher and the LLM:

1. Call `find_matches()` — resolves exact and probable pairs.
2. If unmatched items remain, send them to the LLM with `client.beta.chat.completions.parse()`.
3. The LLM returns a structured `_UnmatchedClassification` with one `UnmatchedItem` per exception.
4. Compute `reconciling_items_value` from timing-difference items only.
5. Determine `is_reconciled`: `|bank_balance - gl_balance - reconciling_value| <= 0.01`.

In [ ]:
import json
from openai import OpenAI

RECON_SYSTEM = """You are a senior bank reconciliation specialist.
Classify each unmatched item into one of:
- timing_difference: deposit in transit or outstanding cheque
- bank_charge: bank fee with no GL counterpart
- duplicate: same amount+date appearing twice
- missing_booking: bank entry with no GL counterpart
- fraud_indicator: unusual item with no business explanation
Provide a specific recommended_action for each item."""


class _UnmatchedClassification(BaseModel):
    items: List[UnmatchedItem]


def run(
    bank_txns: List[Dict],
    gl_entries: List[Dict],
    period: str,
    bank_closing_balance: float,
    gl_cash_balance: float,
    model: str = "gpt-4.1-nano",
) -> BankReconciliationSummary:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    matched_raw, unmatched_bank, unmatched_gl = find_matches(bank_txns, gl_entries)
    matched_pairs = [MatchedPair(**m) for m in matched_raw]

    unmatched_items: List[UnmatchedItem] = []
    if unmatched_bank or unmatched_gl:
        payload = {"unmatched_bank_transactions": unmatched_bank, "unmatched_gl_entries": unmatched_gl}
        user_msg = f"Period: {period}\nBank: {bank_closing_balance}\nGL: {gl_cash_balance}\n\n{json.dumps(payload, indent=2)}"
        resp = client.beta.chat.completions.parse(
            model=model,
            messages=[{"role": "system", "content": RECON_SYSTEM}, {"role": "user", "content": user_msg}],
            response_format=_UnmatchedClassification,
        )
        parsed = resp.choices[0].message.parsed
        if parsed:
            unmatched_items = parsed.items

    reconciling_value = sum(
        item.amount if item.source == "bank" else -item.amount
        for item in unmatched_items
        if item.exception_type == "timing_difference"
    )
    gap = abs(bank_closing_balance - gl_cash_balance - reconciling_value)
    is_reconciled = gap <= 0.01
    note = (
        f"Reconciliation complete for {period}." if is_reconciled
        else f"Unexplained difference of {gap:,.2f} remains for {period}."
    )

    return BankReconciliationSummary(
        period=period,
        bank_closing_balance=bank_closing_balance,
        gl_cash_balance=gl_cash_balance,
        matched_pairs=matched_pairs,
        unmatched_items=unmatched_items,
        reconciling_items_value=reconciling_value,
        is_reconciled=is_reconciled,
        reconciliation_note=note,
    )


print("run() defined.")

## Part 5 — Run Scenario 1: Standard June 2025 Close

In [ ]:
BANK_S1 = [
    {"txn_id": "B01", "date": "2025-06-02", "description": "Customer payment – Invoice 1041", "debit": 0.0, "credit": 5200.00},
    {"txn_id": "B02", "date": "2025-06-04", "description": "Supplier payment – PO-8821", "debit": 3100.00, "credit": 0.0},
    {"txn_id": "B03", "date": "2025-06-07", "description": "Customer payment – Invoice 1042", "debit": 0.0, "credit": 8750.00},
    {"txn_id": "B04", "date": "2025-06-10", "description": "Payroll run June W1", "debit": 12400.00, "credit": 0.0},
    {"txn_id": "B05", "date": "2025-06-12", "description": "Customer payment – Invoice 1043", "debit": 0.0, "credit": 4620.00},
    {"txn_id": "B06", "date": "2025-06-14", "description": "Rent payment June", "debit": 3500.00, "credit": 0.0},
    {"txn_id": "B07", "date": "2025-06-18", "description": "Customer payment – Invoice 1044", "debit": 0.0, "credit": 6930.00},
    {"txn_id": "B08", "date": "2025-06-21", "description": "Utilities payment", "debit": 820.00, "credit": 0.0},
    {"txn_id": "B09", "date": "2025-06-25", "description": "Customer payment – Invoice 1045", "debit": 0.0, "credit": 2980.00},
    {"txn_id": "B10", "date": "2025-06-30", "description": "Deposit in transit – Invoice 1046", "debit": 0.0, "credit": 1250.00},
    {"txn_id": "B11", "date": "2025-06-30", "description": "Monthly account service fee", "debit": 18.50, "credit": 0.0},
    {"txn_id": "B12", "date": "2025-06-28", "description": "Supplier payment – PO-8830", "debit": 7680.00, "credit": 0.0},
]

GL_S1 = [
    {"entry_id": "GL01", "date": "2025-06-02", "reference": "JE-2001", "amount": 5200.00, "description": "AR receipt Invoice 1041"},
    {"entry_id": "GL02", "date": "2025-06-04", "reference": "JE-2002", "amount": -3100.00, "description": "AP payment PO-8821"},
    {"entry_id": "GL03", "date": "2025-06-07", "reference": "JE-2003", "amount": 8750.00, "description": "AR receipt Invoice 1042"},
    {"entry_id": "GL04", "date": "2025-06-10", "reference": "JE-2004", "amount": -12400.00, "description": "Payroll June W1"},
    {"entry_id": "GL05", "date": "2025-06-12", "reference": "JE-2005", "amount": 4620.00, "description": "AR receipt Invoice 1043"},
    {"entry_id": "GL06", "date": "2025-06-14", "reference": "JE-2006", "amount": -3500.00, "description": "Rent June"},
    {"entry_id": "GL07", "date": "2025-06-18", "reference": "JE-2007", "amount": 6930.00, "description": "AR receipt Invoice 1044"},
    {"entry_id": "GL08", "date": "2025-06-21", "reference": "JE-2008", "amount": -820.00, "description": "Utilities June"},
    {"entry_id": "GL09", "date": "2025-06-25", "reference": "JE-2009", "amount": 2980.00, "description": "AR receipt Invoice 1045"},
    {"entry_id": "GL10", "date": "2025-06-27", "reference": "JE-2010", "amount": -7680.00, "description": "AP payment PO-8830"},
    {"entry_id": "GL11", "date": "2025-06-30", "reference": "JE-2011", "amount": -482.00, "description": "Accrual reversal – prepaid insurance"},
]

summary = run(BANK_S1, GL_S1, "June 2025", bank_closing_balance=42350.00, gl_cash_balance=41100.00)

print(f"Period            : {summary.period}")
print(f"Bank balance      : {summary.bank_closing_balance:,.2f}")
print(f"GL balance        : {summary.gl_cash_balance:,.2f}")
print(f"Matched pairs     : {len(summary.matched_pairs)}")
print(f"Unmatched items   : {len(summary.unmatched_items)}")
print(f"Reconciling value : {summary.reconciling_items_value:,.2f}")
print(f"Is reconciled     : {summary.is_reconciled}")
print(f"Note              : {summary.reconciliation_note}")
for item in summary.unmatched_items:
    print(f"  [{item.item_id}] {item.source} | {item.amount:,.2f} | {item.exception_type}")
    print(f"    Action: {item.recommended_action}")

## Part 6 — Exercise

**Question**: Suppose the bank charges a `Monthly account service fee` of **$18.50**
every month. After three months you notice the fee has never been recorded in the GL.

1. What `exception_type` would each monthly fee receive from the reconciliation agent?
2. What `recommended_action` should the accountant take for the *accumulated* three months?
3. If the fee is expected to recur indefinitely, what process change would prevent
   it from appearing as an exception each month?

*Write your answers below, then check against the answer key.*

In [ ]:
# Your answers here
q1_exception_type = ""  # e.g. 'bank_charge'
q2_recommended_action = ""
q3_process_change = ""

print("Fill in your answers above, then run the answer-key cell below.")

In [ ]:
# Answer key
print("=" * 55)
print("ANSWER KEY")
print("=" * 55)
print()
print("Q1: exception_type = 'bank_charge'")
print("    The fee is applied by the bank with no automatic GL counterpart.")
print("    It is not a timing difference — the bank has processed it and")
print("    it will never appear in the GL on its own.")
print()
print("Q2: recommended_action (3 months accumulated)")
print("    Post a single catch-up journal entry: Dr Bank Charges 55.50 / Cr Cash 55.50")
print("    (3 x 18.50). Reference the three reconciliation periods in the memo.")
print("    Then post a correcting entry for each period to align the P&L correctly.")
print()
print("Q3: process change to prevent recurrence")
print("    Set up a standing monthly journal entry (recurring accrual) in the GL")
print("    that debits Bank Charges and credits Cash for $18.50 on the last day")
print("    of each month. This pre-books the expected charge so the reconciliation")
print("    agent matches it as 'probable' rather than flagging it as a bank_charge.")
print("    Alternatively, configure the bank feed integration to auto-post known")
print("    recurring charges to the correct GL account upon receipt.")